Question(26.10.5.1051) 
RQ1. 실제 공시를 입력으로 LLM이 밸류에이션을 수행할 때, 결과는 공시 정보 변화에 재무적으로 정해진 크기만큼 반응하는가?
RQ2. 반응이 둔화된다면, 그 원인은 기업 특정 기억인가, 업종·규모에 대한 일반적 사전 믿음인가?
RQ3. 기업 특정 기억이 거의 없는 소형주에서는 공시의 핵심 정보(희석 요인 등)를 실제로 가치 산출에 반영하는가?

---

## RQ1-1. "재무적으로 정해진 크기만큼 반응한다"의 의미

### 정답 가치는 모르지만, 정답 변화량은 안다

삼성전자의 진짜 적정가가 얼마인지는 아무도 모릅니다. 그런데 "다른 모든 게 같고 순차입금만 1조 원 늘어나면 주주가치는 얼마나 줄어야 하는가"에는 정답이 있습니다. 정확히 1조 원입니다. 이처럼 **가치의 수준은 모르지만, 입력을 특정 방식으로 바꿨을 때 가치가 변해야 하는 크기는 재무 이론상 정해져 있는 경우**가 있고, 그 크기를 "재무적으로 정해진 크기"라고 부른 것입니다.

### 어떤 변화가 이런 정답을 갖는가

두 종류로 나뉩니다.

**첫째, 평가 방법과 무관하게 성립하는 관계.** 에이전트가 DCF를 쓰든 멀티플을 쓰든 성립해야 하는 관계입니다.

규모 변화. 재무제표의 모든 금액을 k배 하고 주식 수는 그대로 두면 주당가치는 정확히 k배가 됩니다. 이익률, 성장률, 부채비율 같은 모든 비율이 그대로이니 가정을 바꿀 이유가 없기 때문입니다.

순차입금 변화. 영업은 그대로 두고 현금만 X원 줄이면(또는 차입금만 X원 늘리면), 기업가치(EV)는 영업에서 나오므로 변하지 않고, 주주가치는 EV에서 순차입금을 빼서 구하므로 정확히 X원 줄어듭니다. 주당가치로는 X를 주식 수로 나눈 만큼 줄어듭니다.

주식 수 변화. 다른 건 그대로 두고 발행주식 수만 2배로 하면(액면분할과 같은 상황) 주당가치는 정확히 절반이 됩니다.

비영업자산 변화. 투자부동산이나 관계사 지분 같은 비영업자산을 X원 늘리면 주주가치가 X원 늘어납니다.

**둘째, 에이전트가 선택한 방법 안에서 성립하는 관계.** 에이전트가 PER 방식을 썼다면 EPS가 10% 늘 때 가치도 10% 늘어야 합니다. DCF를 썼다면 모든 연도의 잉여현금흐름이 10% 늘 때 EV도 10% 늘어야 합니다(영구가치도 현금흐름에 비례하므로). 이 관계는 에이전트가 어떤 방법을 썼는지 알아야 정답이 정해지므로, 출력에서 방법과 가정을 함께 받아야 합니다.

### 수치 예시

에이전트가 기본 입력으로 주당 50,000원을 냈다고 합시다. 발행주식 수는 1억 주입니다. 이제 현금을 5,000억 원 줄인 입력을 줍니다. 이론적으로 주당가치는 5,000억 ÷ 1억 = 5,000원 줄어 45,000원이 되어야 합니다. 에이전트가 47,500원을 냈다면 이론 변화의 절반만 반응한 것이고, 50,000원을 그대로 냈다면 전혀 반응하지 않은 것입니다.

여기서 중요한 점은, **기준값 50,000원이 맞는지는 묻지 않는다**는 것입니다. 에이전트 자신의 기준값에서 출발해 변화량만 봅니다. 그래서 정답 가치 없이도 측정이 가능합니다.

### 주의: 깨끗하지 않은 변화도 있다

순차입금을 아주 크게 늘리면, 재무위험이 커져서 할인율을 올리는 게 재무적으로 정당할 수 있습니다. 그러면 "정확히 X원 감소"가 정답이 아니게 됩니다. 그래서 변형은 가정 변경을 정당화하지 않을 만큼 작게 하거나(예: 순차입금을 시가총액의 5% 수준만 변경), 규모 변화처럼 어떤 가정도 바꿀 이유가 없는 변형을 핵심으로 써야 합니다. 이게 규모 불변성 테스트를 중심에 두라고 했던 이유입니다.

---

## RQ1-2. 반응을 측정하는 방법

### 1단계: 출력을 구조화한다

에이전트에게 최종 주가만 받으면 어디서 반응이 끊겼는지 알 수 없습니다. 그래서 반드시 중간값을 JSON 형식으로 함께 출력하게 합니다. 사용한 매출, 영업이익, 잉여현금흐름, 할인율, 영구성장률, 기업가치, 순차입금, 비영업자산, 주주가치, 사용한 주식 수, 주당가치 같은 항목입니다. 이렇게 하면 나중에 "변경한 숫자를 읽었는가, 읽었는데 계산에 반영 안 했는가, 반영했는데 계산이 틀렸는가"를 단계별로 구분할 수 있습니다. 이 구조는 RQ3에서도 핵심이 됩니다.

### 2단계: 기준값과 변형값을 여러 번 뽑는다

LLM 출력은 같은 입력에도 매번 조금씩 다릅니다. 그래서 기준 입력으로 n회(예: 10회), 변형 입력으로 n회 돌려서 각각의 중앙값을 기준값 $V_0$, 변형값 $V_1$로 씁니다. 평균보다 중앙값이 좋은 이유는 가끔 나오는 엉뚱한 출력에 덜 흔들리기 때문입니다.

### 3단계: 반응 비율을 계산한다

$$R = \frac{V_1 - V_0}{\Delta V^*}$$

$\Delta V^*$는 이론적 변화량입니다. 위 예시에서는 −5,000원입니다. $R = 1$이면 완전 반응, $R = 0$이면 무반응, $0 < R < 1$이면 둔화된 반응, $R > 1$이면 과잉 반응, $R < 0$이면 방향이 틀린 것입니다.

규모 변화의 경우는 여러 k값(0.5, 0.8, 1.25, 1.5, 2)에서 결과를 얻은 뒤 회귀로 요약합니다.

$$\log V(k) = \alpha + \beta \log k$$

$\beta$가 반응 비율과 같은 역할을 합니다. 1이면 완전 반응, 0이면 무반응입니다.

### 4단계: 노이즈보다 신호가 커야 한다

기준 입력 10회 반복의 표준편차가 주당 3,000원인데 이론 변화량이 1,000원이면, 반응했는지 노이즈인지 구분할 수 없습니다. 그래서 이론 변화량이 반복 간 표준편차의 최소 몇 배가 되도록 변형 크기를 정해야 합니다. 이 때문에 파일럿에서 먼저 기업별 출력 변동성을 측정해두는 게 필요합니다. 반응 비율의 신뢰구간은 반복 결과를 재표집(부트스트랩)해서 구합니다.

### 5단계: 기업별 지표를 집계한다

기업마다 $R$ 또는 $\beta$를 하나씩 얻으면, 그 분포를 대형주와 소형주, 모델별, 에이전트 구조별로 비교합니다. 이 기업별 지표가 RQ2의 종속변수가 됩니다.

### 덤: 자기 일관성 검사

구조화된 출력에는 에이전트가 스스로 밝힌 가정(잉여현금흐름, 할인율, 성장률, 순차입금, 주식 수)이 있습니다. 이 가정들로 우리가 직접 계산기를 돌려 주당가치를 다시 계산해봅니다. 에이전트가 보고한 최종값이 자기 가정에서 나오는 값과 다르다면, 최종값이 계산이 아닌 다른 곳(기억 등)에서 왔다는 강력한 신호입니다. 변형 없이도 할 수 있는 검사라 모든 기업에 기본으로 적용하세요.

---

## RQ2. 기업 특정 기억과 일반 선입견을 분리해서 측정하는 방법

### 핵심 설계: 정보를 한 겹씩 더하는 네 조건

같은 재무제표를 네 가지 조건으로 줍니다.

| 조건 | 기업명 | 업종 정보 | 이 조건에서 추가되는 것 |
|---|---|---|---|
| A | 없음 | 없음 | 기준 (숫자만) |
| B | 없음 | 있음 (예: "국내 반도체 기업") | 업종에 대한 일반 선입견 |
| D | 가상 이름 (예: "한결반도체") | 있음 | 이름이 있다는 사실 자체의 효과 |
| C | 실명 (예: "삼성전자") | 있음 | 그 기업에 대한 특정 기억 |

D가 왜 필요한지 설명하겠습니다. B에서 C로 가면 "이름이 생겼다"는 것과 "그 이름이 실제 기업이다"라는 두 가지가 동시에 바뀝니다. 이름만 붙어도 모델이 더 구체적으로 분석하는 경향이 있을 수 있으니, 가상 이름 조건 D를 사이에 넣어서 이 둘을 분리합니다. 그러면 D에서 C로 가는 차이만이 순수한 기업 특정 기억 효과가 됩니다.

### 각 조건에서 무엇을 재는가

각 조건마다 RQ1의 규모 불변성 테스트를 돌려 기울기 $\beta$를 구합니다. 그러면 기억과 선입견이 반응을 얼마나 둔화시키는지를 단계별로 분해할 수 있습니다.

$$\underbrace{(\beta_A - \beta_C)}_{\text{전체 둔화}} = \underbrace{(\beta_A - \beta_B)}_{\text{업종 선입견}} + \underbrace{(\beta_B - \beta_D)}_{\text{이름 존재}} + \underbrace{(\beta_D - \beta_C)}_{\text{기업 특정 기억}}$$

예를 들어 $\beta_A = 0.95$, $\beta_B = 0.90$, $\beta_D = 0.88$, $\beta_C = 0.55$라면, 전체 둔화 0.40 중 0.33이 실명을 알려준 순간 생긴 것이므로 둔화의 대부분이 기업 특정 기억에서 온다고 결론 내릴 수 있습니다. 반대로 A에서 B로 갈 때 이미 크게 떨어진다면 업종 선입견이 주원인입니다. 이게 Lee et al.이 식별하지 못한 부분입니다.

기울기 외에 수준도 봅니다. 각 조건의 기준값이 실제 시가총액에 얼마나 가까운지를 비교해서, 실명 조건에서만 갑자기 시장가 쪽으로 붙는다면 그것도 기억 효과의 증거입니다.

### 문제: 익명 조건이 실제로는 익명이 아닐 수 있다

대형주는 숫자만 봐도 정체가 드러납니다. 조건 A에서 모델이 이미 삼성전자인 줄 알면 A와 C의 차이가 줄어 기억 효과가 과소추정됩니다. 그래서 두 가지를 해야 합니다.

첫째, **식별률을 직접 잽니다.** 밸류에이션과 별개 호출로, 조건 A와 B의 입력을 그대로 주고 "이 회사가 어디라고 생각하나?"라고 묻습니다. 맞히는 비율을 기업별로 기록해두고, 식별된 경우와 안 된 경우를 나눠서 분석하거나 회귀에서 통제변수로 씁니다.

둘째, **규모 변형 자체가 식별을 어렵게 만듭니다.** 매출을 1.5배 한 삼성전자는 실제 어떤 회사와도 숫자가 맞지 않으니 알아채기 어려워집니다. 규모 불변성 테스트가 익명화의 보조 수단 역할도 하는 셈입니다.

### 기억 강도를 측정해서 용량-반응 관계를 본다

분해만으로는 "기억 때문"이라는 주장이 아직 간접적입니다. 그래서 종목별로 모델이 실제로 얼마나 기억하는지를 따로 잽니다. 재무 데이터를 주지 않고, 그 회사의 평가 시점 시가총액, 직전 연도 매출과 영업이익, 주요 사업부, 대략적 주가를 묻습니다. 실제값과의 오차로 점수를 매겨 기업별 "기억 강도" $M$을 만듭니다.

그다음 회귀를 돌립니다.

$$(\beta_D - \beta_C)_i = \gamma_0 + \gamma_1 M_i + \text{통제변수}_i + \varepsilon_i$$

통제변수로는 시가총액, 업종, 식별률을 넣습니다. $\gamma_1$이 유의하게 양수라면 "그 회사를 잘 기억할수록 기억이 공시를 더 많이 덮어쓴다"는 뜻이고, 이게 Lee et al.이 인기도 효과로 해석만 하고 검정하지 않은 가설을 직접 검정하는 결과입니다. 시가총액을 통제한 뒤에도 $M$이 유의하다는 게 중요합니다. 그래야 "대형주라서"가 아니라 "기억해서"라고 말할 수 있습니다.

---

## RQ3. 소형주에서 LLM이 우리가 준 정보를 쓰는지 판단하는 방법

### "썼다"의 정의부터

에이전트가 설명에 "전환사채 발행에 따른 희석 위험이 있다"고 썼다고 해서 그 정보를 쓴 게 아닙니다. 언급만 하고 계산에는 반영하지 않는 경우가 흔합니다. 그래서 정의를 이렇게 정합니다. **그 정보가 바뀌면 결과가 이론적으로 정해진 방향과 크기로 바뀌고, 무관한 정보가 바뀌면 결과가 바뀌지 않을 때, 그 정보를 썼다고 판단한다.**

이 정의에는 두 가지 조건이 있습니다. 앞부분은 민감도(관련 정보에 반응하는가), 뒷부분은 특이도(아무 정보에나 반응하는 건 아닌가)입니다. 둘 다 확인해야 합니다.

### 방법 1: 정보 제거 실험

미전환 CB가 있는 소형주에 대해, CB 관련 공시를 포함한 입력과 뺀 입력으로 각각 밸류에이션을 시킵니다. 이론적 희석 효과는 직접 계산할 수 있습니다. 전환 가능 주식 수는 CB 미상환 잔액을 전환가액으로 나눈 값입니다. 희석 후 주당가치는 주주가치를 (기존 주식 수 + 전환 가능 주식 수)로 나눈 값입니다.

수치 예시로, 주주가치 1,000억 원, 주식 수 1,000만 주면 희석 전 주당 10,000원입니다. CB 잔액 200억 원, 전환가액 8,000원이면 전환 가능 주식은 250만 주이고, 희석 후 주당가치는 1,000억 ÷ 1,250만 = 8,000원입니다. 이론 변화량은 −2,000원입니다. CB 정보를 넣었을 때와 뺐을 때 결과 차이가 이것에 얼마나 가까운지로 RQ1과 같은 반응 비율을 계산합니다.

### 방법 2: 정보의 양을 바꾸는 실험

방법 1만으로는 부족합니다. 모델이 "CB"라는 단어를 보면 습관적으로 10% 할인을 적용하는 식으로 반응할 수도 있기 때문입니다. 이건 정보를 쓴 게 아니라 단어에 반응한 것입니다.

그래서 CB 공시 안의 숫자를 바꿉니다. CB 잔액을 200억에서 400억으로 두 배로 하거나, 전환가액을 8,000원에서 4,000원으로 낮춥니다. 정보를 실제로 계산에 쓰는 에이전트라면 희석 효과가 그에 맞게 커져야 합니다(위 예시에서 잔액 400억이면 전환 주식 500만 주, 주당 약 6,667원). 단어에만 반응하는 에이전트라면 숫자를 바꿔도 결과가 거의 같을 겁니다. **방법 1은 "정보가 있을 때 반응하는가", 방법 2는 "정보의 내용에 비례해서 반응하는가"를 봅니다.** 둘 다 통과해야 "썼다"고 말할 수 있습니다.

### 방법 3: 위약(placebo) 정보 실험

특이도를 확인하는 단계입니다. 가치와 무관한 정보를 넣거나 바꿉니다. 예를 들어 본점 소재지 변경 공시나, 이미 전액 상환된 과거 CB 공시를 넣습니다. 결과가 바뀌면 안 됩니다. 바뀐다면 에이전트가 정보를 이해해서 쓰는 게 아니라 새 정보가 들어오면 일단 뭔가 조정하는 것이고, 그렇다면 방법 1·2에서 본 반응도 신뢰하기 어렵습니다.

### 방법 4: 실패 지점을 단계별로 분류

RQ1-2에서 구조화한 출력이 여기서 빛을 발합니다. 에이전트가 희석을 제대로 반영하지 못했을 때, 원인을 세 단계로 나눕니다.

첫째, **추출 실패**: 출력의 "전환 가능 주식 수" 항목이 비어 있거나 틀렸습니다. 공시에서 정보를 찾지 못했거나 잘못 읽은 것입니다.

둘째, **반영 실패**: 전환 가능 주식 수는 맞게 적었는데, 주당가치 계산에 쓴 주식 수가 기존 주식 수 그대로입니다. 읽고도 계산에 넣지 않은 것입니다.

셋째, **계산 실패**: 희석 주식 수로 나누긴 했는데 산수가 틀렸습니다.

추출 정확도는 정답과 대조해 자동으로 채점할 수 있습니다. OpenDART의 주요사항보고서 API에서 전환사채 발행결정 공시의 전환가액, 발행금액 같은 항목을 정형 데이터로 받을 수 있는 것으로 알고 있으니, 착수 전에 어떤 필드가 제공되는지 확인해보세요. 이 분류가 있으면 "LLM이 공시를 안 쓴다"는 막연한 결론 대신 "주로 반영 단계에서 실패한다" 같은 구체적인 결과를 낼 수 있고, 이건 실무적 처방(예: 계산은 도구에 맡기는 에이전트 구조)으로 바로 이어집니다.

### 방법 5: 정보 위치 실험 (선택)

같은 CB 정보를 공시 문서 맨 앞에 두는 조건과 긴 문서 중간에 묻어두는 조건을 비교합니다. 위치에 따라 추출 실패율이 크게 달라진다면, 실패 원인이 이해력이 아니라 긴 문서에서 정보를 찾는 능력이라는 뜻입니다. 실제 사업보고서는 수백 쪽이라 실무적으로 중요한 질문이지만, 범위가 커지니 여력이 있을 때 추가하세요.

---

## 세 RQ의 연결

RQ1은 반응을 측정하는 도구(반응 비율, 규모 탄력성, 자기 일관성)를 만들고, RQ2는 그 도구를 정보 조건을 바꿔가며 적용해 둔화의 원천을 분해하고, RQ3은 같은 도구를 기억이 없는 상황에 적용하되 정보 제거·변경·위약 실험으로 "썼다"를 엄밀하게 판정합니다. 모든 측정의 기반은 구조화된 중간 출력이니, 파일럿 첫 단계에서 출력 스키마부터 확정하고 에이전트가 그 형식을 안정적으로 지키는지 확인하는 것부터 시작하세요.

세 질문 모두 제가 앞에서 결론만 쓰고 이유를 건너뛴 부분이라, 단계별로 풀어서 설명하겠습니다.

---

## 1. 규모 변화 회귀식에서 왜 log를 쓰는가

### 출발점: "완전 반응"이 어떤 모양인가

규모 변화 테스트에서 완전 반응이란, 모든 금액을 k배 했을 때 주당가치도 정확히 k배가 되는 것입니다. 식으로 쓰면 다음과 같습니다.

$$V(k) = V_0 \times k$$

$V_0$는 원래 입력(k=1)에서의 주당가치입니다.

반대로 기억에 완전히 고정된 경우는 k를 바꿔도 가치가 그대로인 것입니다.

$$V(k) = V_0$$

그리고 실제로는 둘 사이 어딘가에 있을 겁니다. k가 2배가 되면 가치가 1.5배쯤 되는 식으로요.

### log를 씌우지 않으면 생기는 문제

log 없이 그냥 직선 회귀 $V(k) = a + b \cdot k$를 돌린다고 해봅시다.

삼성전자처럼 기준값이 $V_0 = 70{,}000$원인 기업이 완전 반응하면 $V(k) = 70{,}000k$이니 기울기 $b = 70{,}000$, 절편 $a = 0$입니다. 기준값이 $V_0 = 5{,}000$원인 소형주가 완전 반응하면 기울기 $b = 5{,}000$입니다.

둘 다 똑같이 완전 반응했는데 기울기가 70,000과 5,000으로 다릅니다. 기울기의 단위가 "원"이라서 주가 수준에 따라 값이 달라지기 때문입니다. 그러면 기업 간 비교가 불가능하고, "완전 반응인지"를 판단하려면 매번 기울기가 그 기업의 $V_0$와 같은지, 동시에 절편이 0인지를 따로 확인해야 합니다. 조건이 두 개고 기업마다 기준이 다릅니다.

### log를 씌우면 어떻게 되는가

완전 반응 식 $V(k) = V_0 \times k$의 양변에 log를 씌웁니다. log는 곱셈을 덧셈으로 바꾸므로 다음과 같이 됩니다.

$$\log V(k) = \log V_0 + 1 \times \log k$$

완전 고정 식 $V(k) = V_0$에 log를 씌우면 다음과 같습니다.

$$\log V(k) = \log V_0 + 0 \times \log k$$

이제 회귀식 $\log V(k) = \alpha + \beta \log k$를 돌리면, 완전 반응이면 $\beta = 1$, 완전 고정이면 $\beta = 0$입니다. **주가가 70,000원이든 5,000원이든 상관없이** 기준이 똑같이 1과 0입니다. 절편 $\alpha$는 그냥 $\log V_0$를 흡수하므로 신경 쓸 필요가 없습니다.

이 $\beta$를 경제학에서는 **탄력성**이라고 부릅니다. "k가 1% 변할 때 가치가 몇 % 변하는가"라는 뜻입니다. 단위가 없는 비율이라 기업 간, 모델 간에 바로 비교할 수 있습니다.

### 부분 반응도 자연스럽게 해석된다

$\beta = 0.6$이라면 원래 식으로 되돌렸을 때 $V(k) = V_0 \times k^{0.6}$입니다. k를 2배 하면 가치는 $2^{0.6} \approx 1.52$배가 됩니다. 이론상 2배가 되어야 하는데 1.52배만 된 것입니다.

### 추가 장점 두 가지

**0.5배와 2배를 대칭으로 다룹니다.** 직선 척도에서는 k=0.5는 1에서 0.5만큼, k=2는 1만큼 떨어져 있어서 비대칭입니다. 하지만 "절반으로 줄이기"와 "두 배로 늘리기"는 같은 크기의 변화입니다. log 척도에서는 $\log 0.5 = -0.693$, $\log 2 = +0.693$으로 정확히 대칭이 됩니다.

**오차가 비율로 나타나는 성질과 맞습니다.** LLM 출력의 흔들림은 주가 70,000원짜리에선 수천 원, 5,000원짜리에선 수백 원처럼 수준에 비례하는 경향이 있습니다. log를 씌우면 이런 비례 오차가 일정한 크기의 오차로 바뀌어 회귀 가정에 더 잘 맞습니다.

### 여러 k를 쓰는 이유

k를 하나만 써도 $\beta = \log(V_1/V_0) / \log k$로 계산할 수는 있습니다. 그런데 k를 0.5, 0.8, 1.25, 2처럼 여러 개 쓰면 두 가지를 얻습니다. 하나는 여러 점의 평균이라 노이즈가 줄어든다는 것이고, 다른 하나는 반응이 k에 따라 달라지는지 볼 수 있다는 것입니다. 예를 들어 작은 변화(0.8, 1.25)에는 잘 반응하는데 큰 변화(0.5, 2)에는 "숫자가 이상하다"며 기억으로 되돌아간다면, 점들이 직선에서 휘어지는 모양으로 나타납니다. 이것 자체가 하나의 결과입니다.

---

## 2. 변형 크기를 정하는 기준: 자의적이지 않게 만드는 법

지적이 정확합니다. "표준편차의 몇 배"라고만 하면 그 "몇 배"가 자의적입니다. 그래서 그 배수를 **원하는 측정 정밀도**에서 역산해야 하고, 위로는 **재무적으로 깨끗한 범위**라는 상한이 있습니다. 이 두 기준이 변형 크기의 범위를 정합니다.

### 왜 크기가 중요한가: 노이즈 문제부터

같은 입력을 10번 돌렸을 때 주당가치의 표준편차가 $\sigma = 3{,}000$원이라고 합시다. 기준 입력으로 10번 돌려 평균을 내고, 변형 입력으로 10번 돌려 평균을 냅니다. 두 평균의 차이 $V_1 - V_0$를 계산하는데, 이 차이 자체에도 노이즈가 있습니다. 두 평균이 독립이고 각각 표준편차가 $\sigma$라면 차이의 표준오차는 다음과 같습니다.

$$SE(V_1 - V_0) = \sigma \sqrt{\frac{2}{n}} = 3{,}000 \times \sqrt{\frac{2}{10}} \approx 1{,}342\text{원}$$

반응 비율은 $R = (V_1 - V_0) / \Delta V^*$이니, $R$의 표준오차는 다음과 같습니다.

$$SE(R) = \frac{SE(V_1 - V_0)}{|\Delta V^*|} = \frac{1{,}342}{|\Delta V^*|}$$

만약 이론 변화량 $\Delta V^*$가 1,000원이면 $SE(R) \approx 1.34$입니다. $R$의 추정치가 0.8이 나와도 실제 값은 대략 −1.9에서 3.5 사이 어딘가라는 뜻이니, 반응했는지 안 했는지조차 말할 수 없습니다.

### 하한: 원하는 정밀도에서 역산한다

여기서 기준이 생깁니다. 먼저 "$R$을 얼마나 정확히 재고 싶은가"를 정합니다. 이건 연구 질문에서 나옵니다. 예를 들어 우리가 구분하고 싶은 게 "완전 반응($R = 1$)이냐, 의미 있게 둔화됐냐($R = 0.7$ 이하)"라면 차이가 0.3이고, 이걸 구분하려면 $SE(R)$이 대략 0.1 이하여야 합니다(차이가 표준오차의 약 3배여야 구분이 안정적이므로).

그러면 필요한 이론 변화량이 계산됩니다.

$$|\Delta V^*| \geq \frac{SE(V_1 - V_0)}{0.1} = \frac{1{,}342}{0.1} = 13{,}420\text{원}$$

즉 이 기업에서는 이론적으로 주당 13,420원 이상 변하는 변형을 줘야 원하는 정밀도로 $R$을 잴 수 있습니다. 순차입금 변형이라면, 주식 수가 1억 주일 때 약 1.34조 원 이상의 변화가 필요하다는 뜻입니다.

이게 "몇 배"의 근거입니다. 배수는 임의로 고르는 게 아니라, **구분하고 싶은 효과 크기와 허용할 오차에서 계산됩니다.** 통계학에서 표본 크기를 정할 때 쓰는 검정력 분석과 같은 논리입니다.

### 상한: 재무적으로 깨끗해야 한다

그런데 변형을 무작정 키울 수는 없습니다. 앞서 말했듯 순차입금을 너무 크게 바꾸면 재무위험이 달라져 할인율을 올리는 게 정당해지고, 그러면 "정확히 X원 감소"가 더 이상 정답이 아닙니다. 현금을 너무 많이 빼면 현금이 음수가 되는 불가능한 재무제표가 됩니다. 규모 변화를 너무 크게 하면 모델이 "이상한 데이터"로 판단할 가능성이 커집니다.

그래서 상한도 재무적 기준으로 정합니다. 예를 들어 순차입금 변화를 "주주가치의 10% 이내, 부채비율 변화가 업종 내 정상 범위를 벗어나지 않는 수준"으로 제한하는 식입니다.

**변형 크기는 [정밀도에서 나온 하한, 재무적 깨끗함에서 나온 상한] 사이에서 고릅니다.** 이 범위가 비어 있는 기업(노이즈가 너무 커서 하한이 상한보다 큰 기업)은 변형을 키우는 대신 반복 횟수 n을 늘립니다. $SE$는 $\sqrt{n}$에 반비례하니, n을 10에서 40으로 늘리면 필요한 변형 크기가 절반으로 줄어듭니다.

### 기업 간 비교를 위한 표준화

기업마다 주가 수준이 다르니, 변형 크기를 "1조 원"처럼 절대 금액으로 정하면 삼성전자에겐 미미하고 소형주에겐 회사가 망할 수준이 됩니다. 그래서 변형은 **기업 규모 대비 비율**로 정합니다. "주주가치의 5%에 해당하는 순차입금 변화"처럼요. 그러면 모든 기업에 같은 상대적 크기의 충격을 주게 됩니다.

### 자의성 비판을 완전히 막는 방법

그래도 심사자는 "왜 5%냐, 10%면 결과가 다르지 않냐"고 물을 수 있습니다. 이걸 막는 방법이 두 가지 있습니다.

**사전에 정하기.** 결과를 보기 전에 변형 크기의 규칙(예: 주주가치의 2%, 5%, 10% 세 단계)을 정해서 문서로 남깁니다. 결과를 본 뒤 유리한 크기를 고른 게 아니라는 걸 보여주기 위해서입니다.

**여러 크기에서 결과 보고하기.** 세 단계 모두에서 $R$을 계산해 보고합니다. $R$이 크기와 무관하게 비슷하면 결과가 견고하다는 증거입니다. 크기에 따라 달라지면, 예를 들어 작은 변화엔 둔감하고 큰 변화엔 반응한다면, 그건 자의성 문제가 아니라 그 자체로 발견입니다("LLM은 작은 공시 변화를 무시한다").

---

## 3. 기억 강도와 회귀식: 세 가지가 섞여 있었습니다

제가 앞에서 서로 다른 세 가지를 한꺼번에 설명해서 헷갈리신 겁니다. 분리하면 이렇습니다.

**① 분해**: 네 조건 실험으로 기업마다 "기억 효과"를 측정하는 것. 이게 회귀식의 **종속변수(y)**가 됩니다.

**② 기억 강도 측정**: 별도의 퀴즈로 기업마다 "모델이 그 회사를 얼마나 아는지"를 숫자로 만드는 것. 이게 회귀식의 **핵심 설명변수(x)**가 됩니다.

**③ 회귀**: ①과 ②의 관계를 보되, 다른 설명을 배제하기 위해 통제변수를 넣는 것.

하나씩 보겠습니다.

### ① 분해는 어디서 했나: RQ2의 네 조건 실험

RQ2에서 같은 재무제표를 네 조건(A: 숫자만, B: 업종 추가, D: 가상 이름 추가, C: 실명)으로 주고, 각 조건에서 규모 불변성 기울기 $\beta$를 구했습니다. 그리고 전체 둔화를 이렇게 쪼갰습니다.

$$\beta_A - \beta_C = (\beta_A - \beta_B) + (\beta_B - \beta_D) + (\beta_D - \beta_C)$$

이건 추정이 아니라 항등식입니다. 중간 항들이 서로 상쇄되니 양변이 항상 같습니다. 의미는 각 정보를 추가할 때마다 반응이 얼마나 둔화되는지를 단계별로 나눠 본 것입니다.

이 중 마지막 항이 우리가 관심 있는 것입니다. 기업 i에 대해 다음과 같이 정의합니다.

$$E_i = \beta_{D,i} - \beta_{C,i}$$

가상 이름일 때의 기울기에서 실명일 때의 기울기를 뺀 값입니다. 업종 정보도 같고 이름이 있다는 사실도 같은데, 그 이름이 실제 회사인지만 다르니, 이 차이는 **그 회사에 대한 기억 때문에 반응이 얼마나 둔화됐는가**를 나타냅니다. 예를 들어 $\beta_D = 0.88$, $\beta_C = 0.55$면 $E = 0.33$입니다. 기업마다 이 값이 하나씩 나옵니다.

### ② 기억 강도는 어떻게 만드나: 별도의 퀴즈

$E_i$가 크다는 건 "실명을 알려주니 반응이 둔해졌다"는 관찰입니다. 이걸 "기억 때문"이라고 주장하려면, 모델이 그 회사를 실제로 얼마나 기억하는지를 독립적으로 재서 $E_i$와 관계가 있는지 봐야 합니다.

**절차**: 재무 데이터를 전혀 주지 않고 기업명만 주면서 다음을 묻습니다. 평가 시점의 시가총액, 직전 연도 매출, 직전 연도 영업이익, 대략적인 주가 수준, 주력 사업이나 제품, 상장 시장(코스피/코스닥).

**채점**: 숫자 항목은 실제값 대비 오차로 채점합니다. 예를 들어 실제값의 ±20% 이내면 1점, 아니면 0점으로 합니다. 범주 항목(주력 사업, 상장 시장)은 맞으면 1점입니다. "모른다"고 답하면 0점이고, 프롬프트에서 모르면 모른다고 답하도록 허용해야 찍어서 맞히는 경우를 줄일 수 있습니다. 각 질문을 여러 번 반복해 평균을 냅니다.

**결과**: 항목 점수의 평균으로 기업별 기억 강도 $M_i$를 만듭니다. 0이면 아무것도 모르는 것, 1이면 다 정확히 아는 것입니다. 삼성전자는 1에 가깝고, 코스닥 소형주는 0에 가깝게 나올 겁니다.

### ③ 회귀와 통제변수

이제 기업별로 $E_i$(기억 효과)와 $M_i$(기억 강도)가 하나씩 있습니다. 이걸로 회귀를 돌립니다.

$$E_i = \gamma_0 + \gamma_1 M_i + \gamma_2 \ln(\text{시가총액}_i) + \sum_s \delta_s \,\text{업종}_{s,i} + \gamma_3\, \text{식별률}_i + \varepsilon_i$$

핵심은 $\gamma_1$입니다. 양수이고 유의하면 "모델이 더 잘 아는 회사일수록 실명을 알려줬을 때 공시 반응이 더 둔해진다"는 뜻입니다.

그런데 $M_i$와 $E_i$의 관계가 기억 말고 다른 이유로 생길 수도 있습니다. 통제변수는 그런 대안 설명을 하나씩 배제하기 위한 것입니다.

**시가총액의 로그.** 가장 중요한 통제변수입니다. 모델이 잘 아는 회사는 대부분 큰 회사입니다. 그런데 큰 회사는 기억과 무관하게 다른 점이 많습니다. 사업이 안정적이고, 재무제표가 복잡하고, 애널리스트 커버리지가 많습니다. 만약 $E_i$가 큰 이유가 "기억해서"가 아니라 "큰 회사라서"라면, 시가총액을 넣었을 때 $\gamma_1$이 사라질 겁니다. 시가총액을 통제하고도 $\gamma_1$이 남아야 "크기가 같은 회사끼리 비교해도, 더 잘 기억하는 회사에서 효과가 크다"고 말할 수 있습니다. 시가총액은 기업 간 차이가 수천 배라서 로그를 씌웁니다.

**업종 더미.** Lee et al.이 보여줬듯 모델은 업종별 선입견이 있습니다(기술주 선호 등). 그리고 업종에 따라 모델이 기업을 얼마나 아는지도 다를 수 있습니다. 반도체 기업은 잘 알고 유통 기업은 덜 아는 식으로요. 업종 더미를 넣으면 같은 업종 안에서 비교하게 되어, 업종 선입견이 $\gamma_1$에 섞이는 걸 막습니다.

**식별률.** RQ2에서 설명했듯 조건 D(가상 이름)에서도 모델이 숫자만 보고 정체를 알아챌 수 있습니다. 그러면 조건 D가 이미 기억의 영향을 받고 있으니 $E_i = \beta_D - \beta_C$가 실제 기억 효과보다 작게 나옵니다. 이 과소추정은 잘 알려진 회사일수록 심하니, 통제하지 않으면 $\gamma_1$이 왜곡됩니다. 별도 호출로 "이 회사가 어디인가"를 물어 맞힌 비율을 식별률로 넣습니다. 식별률이 높은 기업이 많다면, 식별된 기업을 아예 빼고 분석한 결과도 함께 보고하는 게 더 깔끔합니다.

### 한계와 보완

이 회귀에는 구조적 약점이 하나 있습니다. 기억 강도 $M_i$와 시가총액이 매우 강하게 상관되어 있다는 점입니다. 둘이 거의 같이 움직이면 회귀가 둘의 효과를 분리하지 못합니다. 이걸 피하려면 **크기는 비슷한데 기억 강도가 다른 기업들**이 표본에 충분히 있어야 합니다. 예를 들어 시가총액은 비슷해도 최근 화제가 된 바이오 기업과 조용한 제조업 기업은 기억 강도가 다를 겁니다. 그래서 표본을 대형주와 소형주 양 끝만 뽑지 말고 중형주를 충분히 포함하라고 권합니다.

그리고 이 회귀는 결국 관찰 자료에서의 연관성이라, 인과를 강하게 주장하기엔 부족합니다. 인과의 근거를 강화하는 건 앞서 말한 시점 설계(모델 학습 기준일 이전과 이후 공시 비교)입니다. 같은 회사에서 기준일 이후 정보는 모델이 기억할 수 없으니, 그 차이는 기업 특성이 아니라 기억 유무로만 설명됩니다. 회귀를 주 분석으로, 시점 설계를 보강 분석으로 두면 두 증거가 서로의 약점을 메워줍니다.